## Load & Explore the Data

In [1]:
from datasets import load_dataset

/Users/nikhilvaishnav/Desktop/Pytorch-Sentiment-analysis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 736387.99 examples/s]


In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
sample = dataset["train"][0]

print("Review:", sample["text"][:300])  # first 300 chars
print("Label:", sample["label"])        # 0 = negative, 1 = positive

Review: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h
Label: 0


### Let's build the vocabulary

In [5]:
from collections import Counter
from preprocess import tokenize, encode_and_pad

In [6]:
counter = Counter()
for sample in dataset["train"]:
    counter.update(tokenize(sample["text"]))

print(f"Total unique words found: {len(counter)}")

Total unique words found: 91353


In [7]:
# Keep only top 15,000 most common words
MAX_VOCAB = 15_000

# most_common() returns list of (word, count) sorted by frequency
vocab = {"<PAD>" : 0, "<UNK>": 1}

for word, count in counter.most_common(MAX_VOCAB - 2):
    vocab[word] = len(vocab)

print(f"Vocab size: {len(vocab)}")
print(f"Index of 'the': {vocab['the']}")
print(f"Index of 'movie': {vocab['movie']}")


Vocab size: 15000
Index of 'the': 2
Index of 'movie': 16


## Now convert a review to integers & Padding

In [8]:
# Test encode_and_pad
short_review = "This movie was great"
encoded_padded = encode_and_pad(short_review, vocab, max_len=10)

print("Encoded + Padded:", encoded_padded)
print("Length:", len(encoded_padded))

Encoded + Padded: [11, 16, 3, 79, 0, 0, 0, 0, 0, 0]
Length: 10


## Encode the full dataset

In [ ]:
MAX_LEN = 200  # each review  -> exactly 200 integers

import torch

def prepare_data(split):
    texts  = [encode_and_pad(s["text"], vocab, MAX_LEN)
              for s in dataset[split]]
    labels = [s["label"] for s in dataset[split]]

    # Convert to tensors
    X = torch.tensor(texts,  dtype=torch.long)
    y = torch.tensor(labels, dtype=torch.float)
    return X, y

print("Processing train set...")
X_train, y_train = prepare_data("train")

print("Processing test set...")
X_test,  y_test  = prepare_data("test")

print("X_train shape:", X_train.shape)  
print("y_train shape:", y_train.shape)  

Processing train set...
Processing test set...
X_train shape: torch.Size([25000, 200])
y_train shape: torch.Size([25000])


## DataLoader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
test_dataset  = TensorDataset(X_test,  y_test)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True)

test_loader = DataLoader(test_dataset,
                         batch_size=BATCH_SIZE,
                         shuffle=False)

X_batch, y_batch = next(iter(train_loader))  # grab first batch

print("X_batch shape:", X_batch.shape)  
print("y_batch shape:", y_batch.shape)  

X_batch shape: torch.Size([64, 200])
y_batch shape: torch.Size([64])


## Building the RNN Model

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
import urllib.request
import zipfile
import os

# Download GloVe 6B (822MB zip — will take a few minutes)
url      = "https://nlp.stanford.edu/data/glove.6B.zip"
zip_path = "glove.6B.zip"

if not os.path.exists("glove.6B.100d.txt"):
    print("Downloading GloVe...")
    urllib.request.urlretrieve(url, zip_path)

    print("Extracting...")
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extract("glove.6B.100d.txt")

    os.remove(zip_path)
    print("Done!")
else:
    print("GloVe already downloaded")

Extracting...
Done!


In [ ]:
def load_glove(filepath):
    glove = {}
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            values    = line.split()
            word      = values[0]
            vector    = torch.tensor(
                            [float(v) for v in values[1:]],
                            dtype=torch.float)
            glove[word] = vector
    return glove

print("Loading GloVe vectors...")
glove = load_glove("glove.6B.100d.txt")
print(f"Loaded {len(glove):,} word vectors")
print(f"Each vector size: {glove['the'].shape}")

Loading GloVe vectors...
Loaded 400,000 word vectors
Each vector size: torch.Size([100])


In [ ]:
EMBED_DIM = 100  # must match GloVe dimension (100d)

# Start with random matrix for all words
# shape: (vocab_size, embed_dim)
embedding_matrix = torch.randn(len(vocab), EMBED_DIM) * 0.01

# For every word in OUR vocab — if GloVe knows it,
# replace random vector with GloVe vector
found     = 0
not_found = 0

for word, idx in vocab.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]  # use pretrained vector
        found += 1
    else:
        not_found += 1                       # keep random (rare words)

print(f"Words covered by GloVe: {found:,}")
print(f"Words not in GloVe:     {not_found:,}")
print(f"Coverage: {found/len(vocab)*100:.1f}%")

Words covered by GloVe: 13,958
Words not in GloVe:     1,042
Coverage: 93.1%


In [16]:
from SentimentLSTM import SentimentLSTM
import torch.nn as nn

In [17]:
# Hyperparameters
VOCAB_SIZE = len(vocab)
EMBED_DIM  = 100        # must match GloVe (100d)
HIDDEN_DIM = 128

In [ ]:
model     = SentimentLSTM(VOCAB_SIZE, EMBED_DIM,
                          HIDDEN_DIM, embedding_matrix).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=2)

print(model)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

SentimentLSTM(
  (embedding): Embedding(15000, 100, padding_idx=0)
  (embedding_dropout): Dropout(p=0.4, inplace=False)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.6, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)
Trainable parameters: 631,041


## Training Loop

In [19]:
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct    = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(X_batch)
        loss        = criterion(predictions, y_batch)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()

        total_loss += loss.item()

        preds = torch.sigmoid(predictions) > 0.5
        correct += (preds == y_batch.bool()).sum().item()

    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)

    return avg_loss, accuracy

In [20]:
def evaluate(model, loader, criterion):
  model.eval()
  total_loss = 0
  correct    = 0

  with torch.no_grad():
      for X_batch, y_batch in loader:
          X_batch = X_batch.to(device)
          y_batch = y_batch.to(device)

          predictions = model(X_batch)
          loss        = criterion(predictions, y_batch)

          total_loss += loss.item()

          preds = torch.sigmoid(predictions) > 0.5
          correct += (preds == y_batch.bool()).sum().item()

  avg_loss = total_loss / len(loader)
  accuracy = correct / len(loader.dataset)

  return avg_loss, accuracy

In [ ]:
EPOCHS = 40
best_test_loss = float('inf')
patience_counter = 0
PATIENCE = 3   # stop if no improvement for 3 epochs

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion)

    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:02d}/{EPOCHS} "
          f"| Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2%} "
          f"| Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2%} "
          f"| LR: {current_lr:.6f}")

    # Early stopping check
    if test_loss < best_test_loss:
        best_test_loss   = test_loss
        patience_counter = 0
        
        # Save best model weights
        torch.save(model.state_dict(), "best_model.pt")
        print(f"   Best model saved!")
    else:
        patience_counter += 1
        print(f"   No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n Early stopping at epoch {epoch+1}")
            break

# Load best weights back
model.load_state_dict(torch.load("best_model.pt"))
print("\nBest model restored")

Epoch 01/40 | Train Loss: 0.670 | Train Acc: 59.11% | Test Loss: 0.693 | Test Acc: 52.42% | LR: 0.000500
   Best model saved!
Epoch 02/40 | Train Loss: 0.624 | Train Acc: 66.23% | Test Loss: 0.641 | Test Acc: 62.29% | LR: 0.000500
   Best model saved!
Epoch 03/40 | Train Loss: 0.580 | Train Acc: 71.10% | Test Loss: 0.549 | Test Acc: 74.58% | LR: 0.000500
   Best model saved!
Epoch 04/40 | Train Loss: 0.529 | Train Acc: 74.55% | Test Loss: 0.474 | Test Acc: 78.10% | LR: 0.000500
   Best model saved!
Epoch 05/40 | Train Loss: 0.481 | Train Acc: 77.73% | Test Loss: 0.443 | Test Acc: 80.01% | LR: 0.000500
   Best model saved!
Epoch 06/40 | Train Loss: 0.461 | Train Acc: 79.16% | Test Loss: 0.435 | Test Acc: 80.76% | LR: 0.000500
   Best model saved!
Epoch 07/40 | Train Loss: 0.443 | Train Acc: 80.10% | Test Loss: 0.465 | Test Acc: 79.22% | LR: 0.000500
   No improvement (1/3)
Epoch 08/40 | Train Loss: 0.431 | Train Acc: 80.71% | Test Loss: 0.390 | Test Acc: 82.78% | LR: 0.000500
   Best mo

In [ ]:
# Unfreeze GloVe embeddings now
model.embedding.weight.requires_grad = True

# Very small LR — we don't want to destroy GloVe vectors
# just nudge them toward movie review language
optimizer = torch.optim.Adam(model.parameters(), lr=0.00005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=2)

# Reset early stopping
best_test_loss   = float('inf')
patience_counter = 0
PATIENCE = 3
EPOCHS   = 20

print("Stage 2: Fine-tuning GloVe embeddings...\n")

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    test_loss,  test_acc  = evaluate(model, test_loader, criterion)

    scheduler.step(test_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1:02d}/{EPOCHS} "
          f"| Train Loss: {train_loss:.3f} | Train Acc: {train_acc:.2%} "
          f"| Test Loss: {test_loss:.3f} | Test Acc: {test_acc:.2%} "
          f"| LR: {current_lr:.6f}")

    if test_loss < best_test_loss:
        best_test_loss   = test_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
        print(f"   Best model saved!")
    else:
        patience_counter += 1
        print(f"   No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\n Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_model.pt"))
print("\nBest stage 2 model restored")

Stage 2: Fine-tuning GloVe embeddings...

Epoch 01/20 | Train Loss: 0.286 | Train Acc: 87.96% | Test Loss: 0.306 | Test Acc: 87.08% | LR: 0.000050
   Best model saved!
Epoch 02/20 | Train Loss: 0.276 | Train Acc: 88.26% | Test Loss: 0.312 | Test Acc: 86.92% | LR: 0.000050
   No improvement (1/3)
Epoch 03/20 | Train Loss: 0.273 | Train Acc: 88.32% | Test Loss: 0.306 | Test Acc: 87.14% | LR: 0.000050
   Best model saved!
Epoch 04/20 | Train Loss: 0.268 | Train Acc: 88.63% | Test Loss: 0.304 | Test Acc: 87.21% | LR: 0.000050
   Best model saved!
Epoch 05/20 | Train Loss: 0.265 | Train Acc: 88.84% | Test Loss: 0.305 | Test Acc: 87.19% | LR: 0.000050
   No improvement (1/3)
Epoch 06/20 | Train Loss: 0.264 | Train Acc: 88.96% | Test Loss: 0.307 | Test Acc: 87.39% | LR: 0.000050
   No improvement (2/3)
Epoch 07/20 | Train Loss: 0.260 | Train Acc: 89.27% | Test Loss: 0.307 | Test Acc: 87.38% | LR: 0.000025
   No improvement (3/3)

 Early stopping at epoch 7

Best stage 2 model restored


In [ ]:
import pickle

# Save model weights 
torch.save(model.state_dict(), "models/sentiment_model.pt")
print("Model weights saved-> sentiment_model.pt")

# Save vocab dictionary
with open("models/vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)
print("Vocab saved-> vocab.pkl")

# Save model config (so we can rebuild architecture)
import json

config = {
    "vocab_size": len(vocab),
    "embed_dim":  EMBED_DIM,
    "hidden_dim": HIDDEN_DIM
}

with open("models/model_config.json", "w") as f:
    json.dump(config, f)
print("Config saved-> model_config.json")

 Model weights saved  -> sentiment_model.pt
 Vocab saved  -> vocab.pkl
 Config saved  -> model_config.json


In [ ]:
import os

files = ["models/sentiment_model.pt", "models/vocab.pkl", "models/model_config.json"]

for f in files:
    size = os.path.getsize(f) / 1024  # size in KB
    print(f"  {f}: {size:.1f} KB")

  sentiment_model.pt: 8328.3 KB
  vocab.pkl: 186.7 KB
  model_config.json: 0.1 KB
